# Лабораторная работа 2

Вариант: искусство. X - width_mm, G - десятилетие по acquired

## Задание 1. Подготовка

Подключаем библиотеки и задаём `SEED`, чтобы случайный выбор строк можно было повторить. `numpy` нужен для числовых расчётов, `pandas` — для таблиц, `matplotlib` — для графиков.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)

## Задание 2. Загрузка данных

Загружаем CSV в таблицу pandas и проверяем первые строки, типы данных и количество пропусков. Затем создаём столбец `decade` с десятилетием приобретения.

In [ ]:
artworks = pd.read_csv("records.csv")
artworks.head()

In [ ]:
artworks.info()

In [ ]:
decades = []

for acquired_year in artworks["acquired"]:
    if pd.isna(acquired_year):
        decades.append("Неизвестно")
    else:
        decade = int(acquired_year // 10 * 10)
        decades.append(str(decade))

artworks["decade"] = decades

`decades = []` создаёт пустой список для нового столбца. Цикл берёт год приобретения каждой работы. `pd.isna()` проверяет пропуск: если год пустой, записываем `Неизвестно`. Иначе операция `acquired_year // 10 * 10` переводит год в десятилетие: например, 1995 превращается в 1990. В конце список добавляется в таблицу как столбец `decade`.

width_mm это float, то есть числа. Пустые ячейки стали пропусками NaN, их 381.

Для G сделала столбец decade: например 1995 -> 1990. Если acquired пустой, то пишу "Неизвестно".

## Задание 3. Пропуски

Выбираем случайные 10% заполненных значений `width_mm`, сохраняем их для проверки и заменяем на `NaN`. Затем восстанавливаем одинаковые пропуски тремя способами: средним, медианой и медианой внутри десятилетия.

In [ ]:
known_widths = artworks[artworks["width_mm"].notna()]
missing_rows = known_widths.sample(frac=0.1, random_state=SEED)
missing_index = missing_rows.index
saved_widths = missing_rows["width_mm"].copy()

data_with_missing = artworks.copy()
data_with_missing.loc[missing_index, "width_mm"] = np.nan
print(data_with_missing["width_mm"].isna().sum())

In [ ]:
mean_value = data_with_missing["width_mm"].mean()
median_value = data_with_missing["width_mm"].median()

mean_table = data_with_missing.copy()
median_table = data_with_missing.copy()
decade_median_table = data_with_missing.copy()

mean_table["width_mm"] = mean_table["width_mm"].fillna(mean_value)
median_table["width_mm"] = median_table["width_mm"].fillna(median_value)

for decade in decade_median_table["decade"].unique():
    group_median = decade_median_table.loc[
        decade_median_table["decade"] == decade, "width_mm"
    ].median()
    missing_in_group = (
        (decade_median_table["decade"] == decade)
        & decade_median_table["width_mm"].isna()
    )
    decade_median_table.loc[missing_in_group, "width_mm"] = group_median

In [ ]:
mean_errors = mean_table.loc[missing_index, "width_mm"] - saved_widths
median_errors = median_table.loc[missing_index, "width_mm"] - saved_widths
decade_errors = (
    decade_median_table.loc[missing_index, "width_mm"] - saved_widths
)

print("Среднее: MAE =", np.abs(mean_errors).mean(),
      "MSE =", (mean_errors ** 2).mean())
print("Медиана: MAE =", np.abs(median_errors).mean(),
      "MSE =", (median_errors ** 2).mean())
print("Медиана по десятилетию: MAE =", np.abs(decade_errors).mean(),
      "MSE =", (decade_errors ** 2).mean())

Лучше всего заполнять медианой по десятилетию, у неё меньше и MAE, и MSE. Ширина работ в разные десятилетия разная (в 1850-х в основном маленькие рисунки, а в 2000-х работы больше).

Медиана лучше среднего по MAE, а среднее лучше по MSE. В данных есть очень широкие работы (до 8920 мм), они тянут среднее вверх (326 против медианы 175). В MSE ошибки возводятся в квадрат, поэтому большие ошибки сильно влияют, а в MAE все ошибки считаются одинаково. Поэтому метрики получаются разные.

## Задание 4. Дубликаты

Выбираем случайные 5% строк и добавляем их точные копии. Затем удаляем повторы по всем столбцам, по `id` и по названию произведения.

In [ ]:
data_for_duplicates = artworks.copy()
copies = data_for_duplicates.sample(frac=0.05, random_state=SEED)
data_with_duplicates = pd.concat(
    [data_for_duplicates, copies], ignore_index=True
)

unique_all = data_with_duplicates.drop_duplicates()
unique_by_id = data_with_duplicates.drop_duplicates(subset="id")
unique_by_title = data_with_duplicates.drop_duplicates(subset="title")

print("До копий:", len(data_for_duplicates),
      data_for_duplicates["width_mm"].mean())
print("После копий:", len(data_with_duplicates),
      data_with_duplicates["width_mm"].mean())
print("По всем столбцам:", len(unique_all), unique_all["width_mm"].mean())
print("По id:", len(unique_by_id), unique_by_id["width_mm"].mean())
print("По title:", len(unique_by_title), unique_by_title["width_mm"].mean())

In [ ]:
title_id_counts = data_with_duplicates.groupby("title")["id"].nunique()
titles_with_different_ids = title_id_counts[title_id_counts > 1].index
same_title = data_with_duplicates[
    data_with_duplicates["title"].isin(titles_with_different_ids)
].sort_values("title")

print("Строк с одинаковым title и разными id:", len(same_title))
same_title[["id", "title", "artist", "width_mm"]].head(10)

По всем столбцам и по id удалились только добавленные копии, осталось 8000 строк и среднее как было.

По title удалилось намного больше (осталось 6038), потому что у разных работ бывает одинаковое название, например "[title not known]" или наброски Тёрнера с одинаковыми названиями. У них разные id, то есть это разные работы, и удалять их неправильно.

## Задание 5. Числовые характеристики

Вручную считаем размах, среднее, медиану, дисперсию и стандартное отклонение. После этого сравниваем ручные результаты с NumPy и pandas.

In [ ]:
widths = artworks["width_mm"].dropna().tolist()
widths.sort()
count = len(widths)

minimum = widths[0]
maximum = widths[-1]
width_range = maximum - minimum

total = 0
for width in widths:
    total += width
average = total / count

middle = count // 2
if count % 2 == 1:
    median = widths[middle]
else:
    median = (widths[middle - 1] + widths[middle]) / 2

sum_squared_errors = 0
for width in widths:
    sum_squared_errors += (width - average) ** 2

variance = sum_squared_errors / count
standard_deviation = variance ** 0.5

print("Размах:", width_range)
print("Среднее:", average)
print("Медиана:", median)
print("Дисперсия:", variance)
print("Стандартное отклонение:", standard_deviation)

In [ ]:
width_series = artworks["width_mm"].dropna()
width_array = np.array(width_series)

print("NumPy:")
print("Размах:", np.ptp(width_array))
print("Среднее:", np.mean(width_array))
print("Медиана:", np.median(width_array))
print("Дисперсия:", np.var(width_array))
print("Стандартное отклонение:", np.std(width_array))

print("pandas:")
print("Размах:", width_series.max() - width_series.min())
print("Среднее:", width_series.mean())
print("Медиана:", width_series.median())
print("Дисперсия:", width_series.var(ddof=0))
print("Стандартное отклонение:", width_series.std(ddof=0))

Ручные расчёты совпали с NumPy и pandas, потому что в pandas я указала `ddof=0`. Это дисперсия и стандартное отклонение для всех наблюдений, то есть деление на `n`. По умолчанию pandas использует `n-1`, но здесь это изменено для честного сравнения с ручной формулой и NumPy.

## Задание 6. Фильтрация

Выбираем два десятилетия и границу ширины. Сначала оставляем строки, где выполнены оба условия, затем строки, где выполнено хотя бы одно условие.

In [ ]:
selected_decades = ["1970", "1990"]
width_limit = 500

has_selected_decade = artworks["decade"].isin(selected_decades)
is_wide = artworks["width_mm"] > width_limit

both_conditions = artworks[has_selected_decade & is_wide]
either_condition = artworks[has_selected_decade | is_wide]

print("Оба условия:", len(both_conditions))
print("Хотя бы одно условие:", len(either_condition))

In [ ]:
only_in_either = either_condition[
    ~either_condition.index.isin(both_conditions.index)
]
print("Только во второй таблице:", len(only_in_either))
only_in_either.head(10)

Выбрала десятилетия 1970 и 1990 и границу 500 мм. Во второй таблице строк намного больше, потому что там хватает одного условия: туда попали все работы из 1970 и 1990 (даже узкие) и широкие работы из других десятилетий.

## Задание 7. Порядок операций

Сравниваем два порядка действий:

- сначала отбираем подходящие строки, затем оставляем пять самых широких;
- сначала берём пять самых широких строк всей таблицы, затем применяем фильтр.

Показываем обе таблицы и объясняем, почему результаты могут отличаться.

In [ ]:
top_after_filter = both_conditions.sort_values(
    "width_mm", ascending=False
).head(5)
top_after_filter

In [ ]:
top_before_filter = artworks.sort_values(
    "width_mm", ascending=False
).head(5)
top_before_filter = top_before_filter[
    top_before_filter["decade"].isin(selected_decades)
    & (top_before_filter["width_mm"] > width_limit)
]
top_before_filter

Получилось по-разному. В первом случае 5 строк, а во втором только 2, потому что среди 5 самых широких работ всей таблицы три из 2000-х и 2010-х, а из 1970 и 1990 только две. Так что результат не всегда одинаковый, он совпадёт только если топ-5 всей таблицы подходит под условие.

## Задание 8. Группировка

Для каждого десятилетия считаем количество строк, долю от всей таблицы, число пропусков `width_mm`, среднее и медиану. По этим данным строим горизонтальную диаграмму.

In [ ]:
groups = artworks.groupby("decade")

stats = pd.DataFrame()
stats["count"] = groups.size()
stats["share"] = stats["count"] / len(artworks)
stats["missing"] = stats["count"] - groups["width_mm"].count()
stats["mean"] = groups["width_mm"].mean()
stats["median"] = groups["width_mm"].median()
stats = stats.sort_values("count", ascending=False)
stats

In [ ]:
counts_for_chart = stats["count"].sort_values()

plt.figure(figsize=(9, 6))
plt.barh(counts_for_chart.index, counts_for_chart.values)
plt.xlabel("Количество произведений")
plt.ylabel("Десятилетие приобретения")
plt.title("Количество произведений по десятилетиям")
plt.tight_layout()
plt.show()

Больше всего работ поступило в 1850-х, это больше половины таблицы (почти все в 1856 году). Потом идут 1970, 1990, 1980 и 2000. Меньше всего в 1820, 1830, 1860 и группе "Неизвестно".

Больше всего пропусков ширины в 1880-х (108 из 121).

## Задание 9. Объект и его группа

Добавляем к каждой исходной строке `group_count`, `group_mean` и `delta`. Затем строим график отклонений первых 100 работ и график тех же значений после сортировки по `delta`.

In [ ]:
artworks["group_count"] = artworks["decade"].map(stats["count"])
artworks["group_mean"] = artworks["decade"].map(stats["mean"])
artworks["delta"] = artworks["width_mm"] - artworks["group_mean"]

print("Количество строк:", len(artworks))
artworks[["id", "decade", "width_mm", "group_count", "group_mean", "delta"]].head()

In [ ]:
first_hundred = artworks[artworks["width_mm"].notna()].head(100)

plt.figure(figsize=(10, 5))
plt.scatter(range(1, len(first_hundred) + 1), first_hundred["delta"])
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Порядковый номер произведения")
plt.ylabel("Отклонение от среднего группы")
plt.title("Отклонения первых 100 произведений")
plt.show()

In [ ]:
sorted_by_delta = first_hundred.sort_values("delta")

plt.figure(figsize=(10, 5))
plt.plot(
    range(1, len(sorted_by_delta) + 1),
    sorted_by_delta["delta"],
    marker="o"
)
plt.xlabel("Новый порядковый номер")
plt.ylabel("Отклонение от среднего группы")
plt.title("Отсортированные отклонения")
plt.show()

В таблице осталось 8000 строк, потому что новые столбцы добавлены к исходному DataFrame, а строки не удалялись. `group_count` одинаков для всех работ одного десятилетия, `group_mean` показывает среднюю ширину этого десятилетия, а `delta` показывает отклонение конкретной работы от среднего. На первом графике точка выше нуля означает ширину выше среднего группы. После сортировки `delta` линия не убывает, потому что значения расположены от меньшего к большему.

## Задание 10. Квантили и boxplot

Считаем квантили 0.25, 0.5 и 0.75. Затем выбираем пять самых больших групп и сравниваем распределение ширины с помощью boxplot.

In [ ]:
quantiles = artworks["width_mm"].dropna().quantile([0.25, 0.5, 0.75])
quantiles

In [ ]:
largest_decades = stats.head(5).index
boxplot_data = []

for decade in largest_decades:
    widths_in_group = artworks.loc[
        artworks["decade"] == decade, "width_mm"
    ].dropna()
    boxplot_data.append(widths_in_group)

plt.figure(figsize=(9, 6))
plt.boxplot(
    boxplot_data,
    tick_labels=[str(decade) for decade in largest_decades]
)
plt.xlabel("Десятилетие приобретения")
plt.ylabel("Ширина произведения, мм")
plt.title("Распределение ширины в пяти самых больших группах")
plt.show()

Медиана выше всего у 2000 и 1970, ниже всего у 1850. Сильнее всего разброс у 2000, у неё самый большой ящик, а у 1850 ящик совсем маленький, там почти все работы небольшие.